In [33]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch

import pickle
from pathlib import Path

import sys
base_path = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(base_path / '2_Propensities'))
import MF_class as MF

# 1 Choosing Dataset

In [34]:
datasets = ['ml-1m', 'steam', 'goodreads']
DATASET = datasets[2]

print(f"Using dataset: {DATASET}")

Using dataset: goodreads


# 2 Loading Dataset and Propensities Model

In [35]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
path = base_artifacts / 'Datasets' / 'Processed' / DATASET
train = pd.read_csv(path / 'train.csv')
test = pd.read_csv(path / 'test.csv')
with open(path / 'item_dict.pkl', 'rb') as f:
    item_dict = pickle.load(f)

n_users = len(train['user_id'].unique())
n_items = len(item_dict)

full_data = pd.concat([train, test], ignore_index=True)
title2id = {v: k for k, v in item_dict.items()}

# Load chosen pairs and item dictionary
with open(base_artifacts / 'Chosen_Pairs' / f'{DATASET}_chosen_pairs.pkl', 'rb') as f:
    chosen_pairs = pickle.load(f)

chosen_pairs_ids = [
    (title2id[title_A], title2id[title_B])
    for title_A, title_B in chosen_pairs
]

In [36]:
model_path = base_artifacts / 'Propensity_Models'

with open(model_path / f'MF_params_{DATASET}.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

MF_model = MF.MatrixFactorizationTorch(
    n_users=loaded_params['n_users'], 
    n_items=loaded_params['n_items'], 
    n_factors=loaded_params['n_factors']
)

model_name = f'MF_model_{DATASET}'
MF_model.load(path=model_path / (model_name + '.pt'))
MF_model.eval()

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            7801
Number of items:            6384
Number of factors:          50
Learning rate:              0.001
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           20
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-10 15:46:26


MatrixFactorizationTorch()

# 3 Processing ChatGPT Response

In [37]:
folder = base_artifacts / 'API_Results' / DATASET
csvs_in_folder = [p.name for p in folder.iterdir() if p.is_file() and p.suffix == '.csv']
oracle_file_name = csvs_in_folder[0]
oracle = pd.read_csv(folder / oracle_file_name)

In [38]:
clean_string = lambda s: s[1:-1] if s[0] == "'" and s[-1] == "'" else s
oracle['title_A'] = oracle['title_A'].apply(clean_string)
oracle['title_B'] = oracle['title_B'].apply(clean_string)

known_titles = title2id.keys()
oracle['titles_known'] = oracle['title_A'].apply(lambda x: x in known_titles) & oracle['title_B'].apply(lambda x: x in known_titles)
print(f"Filling {(oracle['titles_known'] == False).sum()} pairs with unknown titles.")

filling_idx = oracle[oracle['titles_known'] == False].index
for idx in filling_idx:
    oracle.loc[idx, 'title_A'] = chosen_pairs[idx][0]
    oracle.loc[idx, 'title_B'] = chosen_pairs[idx][1]

pd.DataFrame(oracle['causal_effect'].value_counts())

Filling 36 pairs with unknown titles.


,count
causal_effect,
0,9187
1,520
3,178
2,115


# 4 Defining Baselines

In [39]:
pivot_real = full_data.pivot(index='user_id', columns='item_id', values='interaction').fillna(0)
itemid_to_colidx_pivot_real = {item_id: col_idx for col_idx, item_id in enumerate(pivot_real.columns)}
pivot_real_np = pivot_real.values

Q_normalized = (MF_model.Q / torch.norm(MF_model.Q, dim=1, keepdim=True)).cpu().detach().numpy()

def cosimilarity(idx1, idx2):
    """Calculate cosine similarity between two items."""
    return np.dot(Q_normalized[idx1], Q_normalized[idx2])

def correlation(idx1, idx2):
    """Calculate correlation between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    if T.std() == 0 or Y.std() == 0:
        return 0
    return np.corrcoef(T, Y)[0, 1]

def diff_of_conditionals(idx1, idx2):
    """Calculate difference of conditionals P(Y|T) - P(Y|~T) between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    p_T = np.clip(np.mean(T), 1e-6, 1-1e-6)
    p_Y = np.mean(Y)
    p_TY = np.mean(T * Y)
    return p_TY / p_T - (p_Y - p_TY) / (1 - p_T)

def jacard_index(idx1, idx2):
    """Calculate Jaccard index between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    intersection = np.sum((T > 0) & (Y > 0))
    union = np.sum((T > 0) | (Y > 0))
    if union == 0:
        return 0
    return intersection / union

In [40]:
baseline_path = base_artifacts / 'Datasets' / 'Evaluated'

with open(baseline_path / 'SASRec' / f'{DATASET}_sasrec_scores.pkl', 'rb') as f:
    sasrec_scores = pickle.load(f)

with open(baseline_path / 'Outcome_Model' / f'{DATASET}_om_scores.pkl', 'rb') as f:
    om_scores = pickle.load(f)

# 5 Defining ATE

In [41]:
test_probs = MF_model.predict_prob(
        torch.tensor(test['user_id'].values, dtype=torch.long),
        torch.tensor(test['item_id'].values, dtype=torch.long)
    )
test_copy = test.copy()
if test_copy['timestamp'].dtype == 'O':
    test_copy['timestamp'] = pd.to_datetime(test_copy['timestamp'], errors='coerce').astype(np.int64) // 10**9
    test_copy['timestamp'] = test_copy['timestamp'].apply(lambda x: x if x > 0 else np.inf)

pivot_test_timestamp = test_copy.pivot(index='user_id', columns='item_id', values='timestamp').fillna(np.inf)
pivot_test_timestamp_np = pivot_test_timestamp.values
itemid_to_colidx = {id: i for i, id in enumerate(pivot_test_timestamp.columns)}

test_copy['probability'] = test_probs.cpu().detach().numpy()
pivot_test_pred = test_copy.pivot(index='user_id', columns='item_id', values='probability')
pivot_test_pred_np = pivot_test_pred.values

In [42]:
test_interaction_time_cols  = {
    item: pivot_test_timestamp_np[:, colidx]
    for item, colidx in itemid_to_colidx.items()
}

pred_cols = {
    item: pivot_test_pred_np[:, colidx]
    for item, colidx in itemid_to_colidx.items()
}

all_interaction_cols = {
    item: pivot_real_np[:, colidx]
    for item, colidx in itemid_to_colidx_pivot_real.items()
}

test_users = test['user_id'].unique()

In [43]:
def get_ATE(
    cause_item,
    effect_item,
    clip=0,
    drop_inverted=True,
    stabilized=True,
    mode="ipw",   # "ipw", "dr", "om"
):
    """
    Estimate ATE using IPW, DR, or OM-only.

    Parameters
    ----------
    mode : {"ipw", "dr", "om"}
        ipw : inverse propensity weighting only
        dr  : doubly robust estimator
        om  : outcome model only (plug-in)
    """

    # --------------------------------------------------
    # Filter inverted interactions
    # --------------------------------------------------

    users        = np.array(test_users)
    cause_times  = test_interaction_time_cols[cause_item]
    effect_times = test_interaction_time_cols[effect_item]
    pi           = pred_cols[cause_item]

    if drop_inverted:
        keep = np.where((cause_times <= effect_times) | (cause_times == np.inf))[0]

        users        = users[keep]
        cause_times  = cause_times[keep]
        effect_times = effect_times[keep]
        pi           = pi[keep]

    T = (cause_times < np.inf).astype(float)
    Y = (effect_times < np.inf).astype(float)

    n = len(T)

    pi = np.clip(pi, clip, 1 - clip)

    # --------------------------------------------------
    # Outcome model (if needed)
    # --------------------------------------------------

    if mode in {"dr", "om"}:
        mu1 = om_scores[(cause_item, effect_item)][1]
        mu0 = om_scores[(cause_item, effect_item)][0]
        if drop_inverted:
            mu1 = mu1[keep]
            mu0 = mu0[keep]
        base = mu1 - mu0
    else:
        base = np.zeros(n)

    # --------------------------------------------------
    # IPW components (if needed)
    # --------------------------------------------------

    if mode in {"dr", "ipw"}:
        D1 = T / pi
        D0 = (1 - T) / (1 - pi)

        if mode == "dr":
            N1 = T * (Y - mu1) / pi
            N0 = (1 - T) * (Y - mu0) / (1 - pi)
        else:  # IPW only
            N1 = Y * D1
            N0 = Y * D0

        mN1 = N1.mean()
        mN0 = N0.mean()
        mD1 = D1.mean()
        mD0 = D0.mean()

        ESS_1 = (D1.sum() ** 2) / (np.sum(D1 ** 2) + 1e-12)
        ESS_0 = (D0.sum() ** 2) / (np.sum(D0 ** 2) + 1e-12)
        ESS_ATE = 2.0 / (1.0 / (ESS_1 + 1e-12) + 1.0 / (ESS_0 + 1e-12))

    # --------------------------------------------------
    # Point estimate
    # --------------------------------------------------

    eps = 1e-12
    if mode == "om":
        ATE = base.mean()

    elif stabilized:
        term_1 = mN1 / (mD1 + eps)
        term_0 = mN0 / (mD0 + eps)
        ATE = base.mean() + term_1 - term_0

    else:
        term_1 = mN1
        term_0 = mN0
        ATE = base.mean() + term_1 - term_0

    # --------------------------------------------------
    # Variance (delta method)
    # --------------------------------------------------

    if mode == "om":
        # simple variance of base
        var_hat = base.var(ddof=1) / n

    elif stabilized:
        Z = np.column_stack([base, N1, D1, N0, D0])
        g = np.array([
            1.0,
            1.0 / (mD1 + eps),
            -mN1 / ((mD1 + eps)**2),
            -1.0 / (mD0 + eps),
            mN0 / ((mD0 + eps)**2),
        ])
        S = np.cov(Z, rowvar=False, ddof=1)
        var_hat = (g @ S @ g) / n

    else:
        Z = np.column_stack([base, N1, N0])
        g = np.array([1.0, 1.0, -1.0])
        S = np.cov(Z, rowvar=False, ddof=1)
        var_hat = (g @ S @ g) / n

    STD = float(np.sqrt(max(var_hat, 0.0)))

    return {
        "ATE": ATE, 
        "STD": STD,
        "size": n,
        "ESS_0": ESS_0 if mode in {"ipw", "dr"} else None,
        "ESS_1": ESS_1 if mode in {"ipw", "dr"} else None,
        "ESS_ATE": ESS_ATE if mode in {"ipw", "dr"} else None
    }

# 6 Generate Results

In [44]:
clip_dict = {
    'ml-1m': 0.05,
    'steam': 0.01,
    'goodreads': 0.01,
}
clip = clip_dict[DATASET]

In [45]:
def process_pair(pair):

    c = pair[0]
    e = pair[1]

    ate_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=clip,
        drop_inverted=True,
        stabilized=False,
        mode="ipw",
    )

    ate_dr_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=clip,
        drop_inverted=True,
        stabilized=False,        
        mode="dr",
    )

    ate_om_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=clip,
        drop_inverted=True,
        stabilized=False,        
        mode="om",
    )
    
    abl_dict = get_ATE(
        cause_item=c, 
        effect_item=e,
        clip=0.5,
        drop_inverted=True,
        stabilized=False,
        mode="ipw",
    )

    ate_stabilized_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=clip,
        drop_inverted=True,
        stabilized=True,
        mode="ipw",
    )

    return {
        "cause_id": pair[0],
        "effect_id": pair[1],
        "ATE": ate_dict["ATE"],
        "STD": ate_dict["STD"],
        "size": ate_dict["size"],
        "ESS_0": ate_dict["ESS_0"],
        "ESS_1": ate_dict["ESS_1"],
        "ESS_ATE": ate_dict["ESS_ATE"],
        "ATE_DR": ate_dr_dict["ATE"],
        "STD_DR": ate_dr_dict["STD"],
        "ATE_OM": ate_om_dict["ATE"],
        "STD_OM": ate_om_dict["STD"],
        "ABLT": abl_dict["ATE"],
        "STD_ABLT": abl_dict["STD"],
        "ATE_STABILIZED": ate_stabilized_dict["ATE"],
        "STD_STABILIZED": ate_stabilized_dict["STD"],
        "cosine_similarity": cosimilarity(*pair),
        "correlation": correlation(*pair),
        "diff_of_conditionals": diff_of_conditionals(*pair),
        "jacard_index": jacard_index(*pair),
        "sasrec_score": sasrec_scores[pair],
    }

In [46]:
all_results = []
for pair in tqdm(chosen_pairs_ids):
    results = process_pair(pair)
    all_results.append(results)

raw_results = pd.DataFrame(all_results)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [47]:
merged = pd.merge(
    left=oracle,
    right=raw_results,
    left_index=True,
    right_index=True,
)

In [48]:
merged.to_csv(base_artifacts / 'Datasets' / 'Evaluated' / f'{DATASET}_evaluated.csv', index=False)